# 人机交互

[人机交互](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)（Human-in-the-loop, HITL）指智能体为了向人类索要执行权限或额外信息而主动中断，并在获得人类反馈后继续执行的过程。

LangGraph 的人机交互功能可以通过内置中间件 HumanInTheLoopMiddleware（HITL）实现。触发人机交互后，HITL 会将当前状态保存到 [checkpointer](https://docs.langchain.com/oss/javascript/langgraph/persistence#checkpointer-libraries) 检查点中，并等待人类回复。获得回复后，再将状态从检查点中恢复出来，继续执行任务。

> **Note**
> 本例只是演示 checkpoint 在人机交互中的作用，无所谓线程结束后记忆是否保持，因此使用内存存储 `InMemorySaver`。在生产环境中，请使用持久化检查点，比如：
>
> - `SqliteSaver`
> - `PostgresSaver`
> - `MongoDBSaver`
> - `RedisSaver`

In [5]:
import os
import uuid  # 用于生成唯一的线程 ID（thread_id）
from dotenv import load_dotenv  # 从 .env 文件加载环境变量
from langchain_openai import ChatOpenAI  # OpenAI 兼容接口的聊天模型
from langchain.agents import create_agent  # 创建 LangGraph Agent 的工厂函数
from langchain.agents.middleware import HumanInTheLoopMiddleware  # 人机交互中间件（HITL）
from langchain.tools import tool  # 将普通函数装饰为 Agent 可调用的工具
from langgraph.checkpoint.memory import InMemorySaver  # 内存中的会话检查点，用于保存/恢复 Agent 状态
from langgraph.types import Command  # 用于向中断的 Agent 发送恢复指令

# 加载 .env 文件中的环境变量（API Key、Base URL 等）
_ = load_dotenv()


下面使用 HITL 中间件，为三种工具配置人工审批流程。

In [6]:
# 配置大模型服务：使用 DashScope（通义千问）的 qwen3-coder-plus 模型
llm = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),  # 从环境变量读取 API 密钥
    base_url=os.getenv("DASHSCOPE_BASE_URL"),  # 从环境变量读取 API 基础 URL
    model="qwen3-coder-plus",  # 模型名称
)

# 工具函数：查询城市天气
@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

# 工具函数：两数相加
@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers and return the sum."""
    return a + b

# 工具函数：计算 BMI（身体质量指数）
@tool
def calculate_bmi(weight_kg: float, height_m: float) -> float:
    """
    Calculate BMI given weight in kg and height in meters.
    BMI = 体重(kg) / 身高(m)^2
    """
    if height_m <= 0 or weight_kg <= 0:
        raise ValueError("height_m and weight_kg must be greater than 0.")
    return weight_kg / (height_m ** 2)

# 创建带工具调用的 Agent，并配置人机交互（HITL）中间件
tool_agent = create_agent(
    model=llm,  # 使用上述配置的大模型
    tools=[get_weather, add_numbers, calculate_bmi],  # 注册三个工具
    middleware=[
        HumanInTheLoopMiddleware(
            # interrupt_on 字典决定哪些工具调用前需要人工审批：
            interrupt_on={
                # 无需触发人工审批：get_weather 工具可直接执行，不中断
                "get_weather": False,
                # 需要审批：调用 add_numbers 前暂停，等待人工审批
                # 设置为 True 表示允许 approve（通过）、edit（编辑）、reject（拒绝）三种审批类型
                "add_numbers": True,
                # 需要审批：调用 calculate_bmi 前暂停，等待人工审批
                # 自定义 allowed_decisions 只允许 approve（通过）和 reject（拒绝），不允许 edit（编辑）
                "calculate_bmi": {"allowed_decisions": ["approve", "reject"]},
            },
            # 审批描述前缀：中断时会显示 "Tool execution pending approval\n\nTool: xxx\nArgs: ..."
            description_prefix="Tool execution pending approval",
        ),
    ],
    checkpointer=InMemorySaver(),  # 启用内存检查点：中断时保存状态，以便审批后恢复
    system_prompt="You are a helpful assistant",  # 系统提示词
)


In [7]:
# 运行 Agent：发送用户请求，触发工具调用和人机交互

# 配置会话线程 ID：uuid.uuid4() 生成唯一标识符
# 同一个 thread_id 可以在多次 invoke 之间保持对话上下文
config = {'configurable': {'thread_id': str(uuid.uuid4())}}

# 第一次 invoke：Agent 接收用户问题，决定调用 calculate_bmi 工具
# 由于 HITL 中间件配置了 calculate_bmi 需要人工审批，Agent 在此处中断
result = tool_agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "我身高180cm，体重180斤，我的BMI是多少"
        # 也可以取消下面这行的注释，测试 get_weather 工具（不需要审批）
        # "content": "what is the weather in sf"
    }]},
    config=config,
)

# 注意：此时 Agent 已中断，不会返回最终结果
# result.get('__interrupt__') 查看中断信息：包含待审批的工具名称、参数等
result.get('__interrupt__')


[Interrupt(value={'action_requests': [{'name': 'calculate_bmi', 'args': {'height_m': 1.8, 'weight_kg': 90}, 'description': "Tool execution pending approval\n\nTool: calculate_bmi\nArgs: {'height_m': 1.8, 'weight_kg': 90}"}], 'review_configs': [{'action_name': 'calculate_bmi', 'allowed_decisions': ['approve', 'reject']}]}, id='3bf560a16f3ccb4b343de245f230bc62')]

从中断信息来看，智能体已触发 `calculate_bmi` 工具调用，并进入等待审批状态。下面我们向智能体发送「审批通过」指令，使其恢复运行。

In [8]:
# 恢复 Agent 执行：向中断发送「审批通过」指令

# 使用 Command 对象恢复中断的 Agent 流程：
# resume={"decisions": [{"type": "approve"}]} 表示批准待执行的工具调用
# 将 "approve" 改为 "reject" 可以拒绝工具调用
# 将 "approve" 改为 "edit" 可以编辑工具参数后再执行（如果 allowed_decisions 包含 "edit"）
result = tool_agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}  # 审批类型：approve（通过）/ edit（编辑）/ reject（拒绝）
    ),
    config=config  # 必须使用相同的 config（thread_id）才能恢复同一个会话
)

# 打印 Agent 的最终回复：calculate_bmi 执行后返回的 BMI 值
result['messages'][-1].content


'您的BMI是27.8。'

> **Note**
> 除了 HITL 之外，LangChain 还提供了多种 **内置中间件**。这里仅列举其中几个，完整列表可以在 [接口文档](https://reference.langchain.com/python/langchain/middleware/#middleware-classes) 中找到。
> 
> | CLASS | DESCRIPTION |
> | --- | --- |
> | SummarizationMiddleware | 在接近 token 上限时自动将对话历史进行摘要 |
> | ModelCallLimitMiddleware | 限制模型调用次数以防止成本过高 |
> | ToolCallLimitMiddleware | 通过限制调用次数来控制工具执行 |
> | ModelFallbackMiddleware | 当主模型失败时自动回退到备用模型 |
> | ...... | ...... |

参考文档：

- [langchain/human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)
- [langchain/short-term-memory](https://docs.langchain.com/oss/python/langchain/short-term-memory)
- [langchain/long-term-memory](https://docs.langchain.com/oss/python/langchain/long-term-memory)
- [langgraph/persistence](https://docs.langchain.com/oss/python/langgraph/persistence)
- [langgraph/use-time-travel](https://docs.langchain.com/oss/python/langgraph/use-time-travel)
- [langgraph/add-memory](https://docs.langchain.com/oss/python/langgraph/add-memory)